In [1]:
%pip install -q transformers accelerate rank-bm25 peft trl datasets pandas tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 996.6/996.6 kB 23.9 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [2]:

import os
from pathlib import Path

BASE_DIR = Path(
    "/kaggle/input/datasets/shubhamdineshjoshi/checkpoints-and-data"
)

print("=" * 75)
print("SEARCH-R1 CHECKPOINTS AND DATA INSPECTION")
print("=" * 75)

if not BASE_DIR.exists():
    print("ERROR: Dataset directory not found!")
    print("Available Kaggle input directories:")
    for p in Path("/kaggle/input").iterdir():
        print(p)
else:
    total_size = 0
    total_files = 0

    for root, dirs, files in os.walk(BASE_DIR):
        level = len(Path(root).relative_to(BASE_DIR).parts)
        indent = "    " * level

        print(f"\n{indent}{Path(root).name}/")

        for filename in sorted(files):
            filepath = Path(root) / filename
            size = filepath.stat().st_size
            total_size += size
            total_files += 1

            print(
                f"{indent}    {filename} "
                f"({size / (1024**2):.2f} MB)"
            )

    print("\n" + "=" * 75)
    print("TOTAL FILES:", total_files)
    print(f"TOTAL SIZE: {total_size / (1024**3):.3f} GB")
    print("=" * 75)


SEARCH-R1 CHECKPOINTS AND DATA INSPECTION

checkpoints-and-data/
    __huggingface_repos__.json (0.00 MB)

    search_rl_project/
        experiment_manifest.json (0.00 MB)
        project_manifest.json (0.00 MB)

        nq_splits/
            test.jsonl (6.56 MB)
            train.jsonl (119.87 MB)
            validation.jsonl (6.72 MB)

        wiki/
            wikipedia_passages.jsonl (354.56 MB)

        retrieval_artifacts/
            e5_100k_embeddings.npy (629.51 MB)
            e5_100k_index.faiss (629.51 MB)

        wiki_100k/
            wikipedia_100k_passages.jsonl (312.68 MB)

TOTAL FILES: 10
TOTAL SIZE: 2.011 GB


In [3]:

import os
import shutil
from pathlib import Path

SOURCE = Path(
    "/kaggle/input/datasets/shubhamdineshjoshi/checkpoints-and-data/search_rl_project"
)

PROJECT_DIR = Path("/kaggle/working/search_rl_project")

PROJECT_DIR.mkdir(parents=True, exist_ok=True)

folders = [
    "nq_splits",
    "wiki",
    "wiki_100k",
    "retrieval_artifacts"
]

for folder in folders:
    src = SOURCE / folder
    dst = PROJECT_DIR / folder

    if src.exists():
        if dst.exists():
            shutil.rmtree(dst)

        shutil.copytree(src, dst)
        print(f"Restored: {folder}")
    else:
        print(f"Missing: {folder}")

for filename in ["project_manifest.json", "experiment_manifest.json"]:
    src = SOURCE / filename
    if src.exists():
        shutil.copy2(src, PROJECT_DIR / filename)
        print(f"Restored: {filename}")

print("\nProject restored.")
print("Project directory:", PROJECT_DIR)
print("Train exists:", (PROJECT_DIR / "nq_splits/train.jsonl").exists())
print("Validation exists:", (PROJECT_DIR / "nq_splits/validation.jsonl").exists())
print("Test exists:", (PROJECT_DIR / "nq_splits/test.jsonl").exists())


Restored: nq_splits
Restored: wiki
Restored: wiki_100k
Restored: retrieval_artifacts
Restored: project_manifest.json
Restored: experiment_manifest.json

Project restored.
Project directory: /kaggle/working/search_rl_project
Train exists: True
Validation exists: True
Test exists: True


In [4]:

import json
from pathlib import Path

PROJECT_DIR = Path("/kaggle/working/search_rl_project")

for filename in ["project_manifest.json", "experiment_manifest.json"]:
    path = PROJECT_DIR / filename

    print(f"\n{'='*60}")
    print(filename)
    print("="*60)

    if path.exists():
        with open(path) as f:
            print(json.dumps(json.load(f), indent=2))
    else:
        print("Not found")

print("\n" + "="*60)
print("OTHER KAGGLE INPUT DATASETS")
print("="*60)

for path in Path("/kaggle/input").rglob("*"):
    if path.is_file() and (
        "wiki-18" in path.name.lower()
        or "wiki_dump" in path.name.lower()
    ):
        print(path)



project_manifest.json
{
  "project": "Search-R1-inspired QA with RL",
  "created_at": "2026-10-30 04:52:05",
  "base_model": "Qwen/Qwen2.5-0.5B-Instruct",
  "nq": {
    "source": "Natural-Questions-Base.csv",
    "source_records": 307373,
    "questions_with_short_answers": 106915,
    "train_records": 96223,
    "validation_records": 5346,
    "test_records": 5346,
    "random_state": 42,
    "split": "90/5/5 after filtering empty short answers"
  },
  "wikipedia": {
    "source": "wikimedia/wikipedia",
    "config": "20231101.en",
    "articles_processed": 50000,
    "passages": 469288,
    "passage_words": 120,
    "overlap_words": 30
  },
  "retrieval": {
    "method": "BM25",
    "top_k": 3,
    "tokenization": "lowercase alphanumeric regex"
  },
  "training": {
    "started": false,
    "previous_checkpoints_used": false
  }
}

experiment_manifest.json
{
  "project": "Search-R1 Resource-Constrained Implementation",
  "retriever": "intfloat/e5-small-v2",
  "embedding_dimension": 

In [5]:
!pip install -q faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 88.1 MB/s eta 0:00:00


In [6]:
import faiss
import torch

print("FAISS version:", faiss.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Not available")

FAISS version: 1.15.1
CUDA available: True
GPU: Tesla T4


In [7]:

import os
import json
import torch
import numpy as np
import faiss
from pathlib import Path

PROJECT_DIR = Path("/kaggle/working/search_rl_project")

TRAIN_PATH = PROJECT_DIR / "nq_splits/train.jsonl"
VAL_PATH = PROJECT_DIR / "nq_splits/validation.jsonl"
TEST_PATH = PROJECT_DIR / "nq_splits/test.jsonl"

WIKI_PATH = PROJECT_DIR / "wiki_100k/wikipedia_100k_passages.jsonl"

EMBED_PATH = PROJECT_DIR / "retrieval_artifacts/e5_100k_embeddings.npy"
INDEX_PATH = PROJECT_DIR / "retrieval_artifacts/e5_100k_index.faiss"

print("=" * 65)
print("1. GPU")
print("=" * 65)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory:", round(
        torch.cuda.get_device_properties(0).total_memory / 1024**3, 2
    ), "GB")

print("\n" + "=" * 65)
print("2. DATASET")
print("=" * 65)

for path in [TRAIN_PATH, VAL_PATH, TEST_PATH]:
    print(path.name, "exists:", path.exists(),
          "size:", round(path.stat().st_size / 1024**2, 2) if path.exists() else "N/A", "MB")

with open(TRAIN_PATH, "r", encoding="utf-8") as f:
    sample = json.loads(next(f))

print("\nSample record:")
print(json.dumps(sample, indent=2)[:1500])

print("\n" + "=" * 65)
print("3. RETRIEVAL ARTIFACTS")
print("=" * 65)

for path in [WIKI_PATH, EMBED_PATH, INDEX_PATH]:
    print(path.name, "exists:", path.exists())

if EMBED_PATH.exists():
    embeddings = np.load(EMBED_PATH, mmap_mode="r")
    print("Embedding shape:", embeddings.shape)

if INDEX_PATH.exists():
    index = faiss.read_index(str(INDEX_PATH))
    print("FAISS vectors:", index.ntotal)
    print("FAISS dimension:", index.d)

print("\n" + "=" * 65)
print("4. PASSAGE SAMPLE")
print("=" * 65)

with open(WIKI_PATH, "r", encoding="utf-8") as f:
    passage = json.loads(next(f))

print("Passage keys:", passage.keys())
print("Passage sample:", str(passage)[:700])

print("\n" + "=" * 65)
print("DIAGNOSTIC COMPLETE")
print("=" * 65)


1. GPU
PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GB

2. DATASET
train.jsonl exists: True size: 119.87 MB
validation.jsonl exists: True size: 6.72 MB
test.jsonl exists: True size: 6.56 MB

Sample record:
{
  "id": "48737",
  "question": "which is the last season of game of thrones",
  "answers": [
    "The eighth"
  ],
  "long_answer": "The eighth and final season of the fantasy drama television series Game of Thrones was announced by HBO in July 2016 . Unlike the first six seasons that each had ten episodes and the seventh that had seven episodes , the eighth season will have only six episodes . Like the previous season , it will largely consist of original content not found currently in George R.R. Martin 's A Song of Ice and Fire series and will also adapt material Martin has revealed to showrunners about the upcoming novels in the series , The Winds of Winter and A Dream of Spring .",
  "split": "train"
}

3. RETRIEVAL ARTIFACTS
wikipedia_100k_passag

In [8]:

import os
import json
import importlib.util
import numpy as np
import faiss
import torch

from pathlib import Path
from tqdm.auto import tqdm

PROJECT_DIR = Path("/kaggle/working/search_rl_project")

WIKI_PATH = PROJECT_DIR / "wiki_100k/wikipedia_100k_passages.jsonl"
INDEX_PATH = PROJECT_DIR / "retrieval_artifacts/e5_100k_index.faiss"

print("===== LIBRARY AVAILABILITY =====")

packages = [
    "transformers",
    "peft",
    "trl",
    "accelerate",
    "sentence_transformers",
    "bitsandbytes"
]

for package in packages:
    print(package, ":", importlib.util.find_spec(package) is not None)

print("\n===== LOADING PASSAGES =====")

WIKI_RECORDS = []

with open(WIKI_PATH, "r", encoding="utf-8") as f:
    for line in tqdm(f):
        WIKI_RECORDS.append(json.loads(line))

print("Passages loaded:", len(WIKI_RECORDS))

print("\n===== LOADING FAISS INDEX =====")

FAISS_INDEX = faiss.read_index(str(INDEX_PATH))

print("Index vectors:", FAISS_INDEX.ntotal)
print("Index dimension:", FAISS_INDEX.d)

assert len(WIKI_RECORDS) == FAISS_INDEX.ntotal, (
    "Passage count and FAISS index count do not match!"
)

print("\nPassage and index counts match.")
print("GPU available:", torch.cuda.is_available())


===== LIBRARY AVAILABILITY =====
transformers : True
peft : True
trl : True
accelerate : True
sentence_transformers : True
bitsandbytes : False

===== LOADING PASSAGES =====


0it [00:00, ?it/s]

Passages loaded: 429744

===== LOADING FAISS INDEX =====
Index vectors: 429744
Index dimension: 384

Passage and index counts match.
GPU available: True


In [9]:

from sentence_transformers import SentenceTransformer
import numpy as np
import torch

E5_MODEL_NAME = "intfloat/e5-small-v2"

print("Loading E5 query encoder...")

E5_MODEL = SentenceTransformer(
    E5_MODEL_NAME,
    device="cuda" if torch.cuda.is_available() else "cpu"
)

def search_e5(query, top_k=3):
    """
    Convert the search query into an E5 embedding,
    retrieve the top-k matching Wikipedia passages using FAISS.
    """

    query_embedding = E5_MODEL.encode(
        ["query: " + query],
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False
    )

    query_embedding = query_embedding.astype("float32")

    scores, indices = FAISS_INDEX.search(
        query_embedding,
        top_k
    )

    results = []

    for score, idx in zip(scores[0], indices[0]):
        if idx < 0:
            continue

        passage = WIKI_RECORDS[int(idx)]

        results.append({
            "id": passage["id"],
            "title": passage["title"],
            "text": passage["text"],
            "score": float(score)
        })

    return results


print("E5 query encoder loaded successfully!")


Loading E5 query encoder...


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/67.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  133MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

E5 query encoder loaded successfully!


In [10]:

test_queries = [
    "which is the last season of game of thrones",
    "Who was the first president of the United States?",
    "What is the capital of France?"
]

for query in test_queries:

    print("\n" + "=" * 80)
    print("QUERY:", query)
    print("=" * 80)

    results = search_e5(query, top_k=3)

    for i, result in enumerate(results, 1):
        print(f"\nPassage {i}")
        print("Title:", result["title"])
        print("Score:", round(result["score"], 4))
        print("Text:", result["text"][:500])



QUERY: which is the last season of game of thrones

Passage 1
Title: Vikings: Valhalla
Score: 0.8366
Text: the end of the original series and would be released on Netflix. The 24-episode series was made by MGM Television, and filmed primarily in Ireland, working from the same Ashford Studios in County Wicklow. The series focuses "on the adventures of Leif Erikson, Freydís Eiríksdóttir, Harald Sigurdsson and the Norman king William the Conqueror". On March 9, 2022, the series was officially renewed for a second and third season. On November 21, 2022, it was announced that the second season would premi

Passage 2
Title: The Crown (season 5)
Score: 0.8325
Text: The fifth season of The Crown, which follows the life and reign of Queen Elizabeth II, was released by Netflix on 9 November 2022. It was the first season of the series to be released following both the death of Prince Philip, Duke of Edinburgh on 9 April 2021 and the death of Queen Elizabeth II on 8 September 2022. Imelda Staunto

In [11]:
%pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
Note: you may need to restart the kernel to use updated packages.


In [12]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, TaskType

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    bias="none"
)

model = get_peft_model(base_model, lora_config)

model.gradient_checkpointing_enable()
model.enable_input_require_grads()
model.config.use_cache = False

model.print_trainable_parameters()

print("GPU:", torch.cuda.get_device_name(0))
print("Model loaded successfully!")

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

trainable params: 1,081,344 || all params: 495,114,112 || trainable%: 0.2184
GPU: Tesla T4
Model loaded successfully!


In [13]:
import json
import re
import string
from pathlib import Path

TRAIN_PATH = Path(
    "/kaggle/working/search_rl_project/nq_splits/train.jsonl"
)

# --------------------------------------------------
# 1. Load the complete training split
# --------------------------------------------------

train_records = []

with open(TRAIN_PATH, "r", encoding="utf-8") as f:
    for line in f:
        record = json.loads(line)
        if record.get("question") and record.get("answers"):
            train_records.append(record)

print("Total training questions:", len(train_records))
print("\nSample:")
print("Question:", train_records[0]["question"])
print("Reference answer:", train_records[0]["answers"])


# --------------------------------------------------
# 2. Text normalization
# --------------------------------------------------

def normalize_text(text):
    text = str(text).lower()
    text = text.translate(str.maketrans("", "", string.punctuation))
    return " ".join(text.split())


# --------------------------------------------------
# 3. Token-level F1
# --------------------------------------------------

def calculate_f1(prediction, reference):
    pred_tokens = normalize_text(prediction).split()
    ref_tokens = normalize_text(reference).split()

    if not pred_tokens or not ref_tokens:
        return 0.0

    common = {}

    for token in pred_tokens:
        common[token] = common.get(token, 0) + 1

    overlap = 0

    for token in ref_tokens:
        if common.get(token, 0) > 0:
            overlap += 1
            common[token] -= 1

    if overlap == 0:
        return 0.0

    precision = overlap / len(pred_tokens)
    recall = overlap / len(ref_tokens)

    return 2 * precision * recall / (precision + recall)


# --------------------------------------------------
# 4. Extract generated tags
# --------------------------------------------------

def extract_tag(text, tag):
    pattern = rf"<{tag}>(.*?)</{tag}>"
    match = re.search(pattern, text, flags=re.IGNORECASE | re.DOTALL)

    if match:
        return match.group(1).strip()

    return ""


def is_abstention(answer):
    answer = normalize_text(answer)

    abstention_phrases = [
        "insufficient evidence",
        "not enough evidence",
        "could not find sufficient evidence",
        "i cannot determine",
        "i dont know",
        "unable to determine"
    ]

    return any(phrase in answer for phrase in abstention_phrases)


# --------------------------------------------------
# 5. Initial evidence-support heuristic
# --------------------------------------------------

def evidence_supports_answer(reference_answers, retrieved_passages):
    evidence = normalize_text(
        " ".join(
            p.get("title", "") + " " + p.get("text", "")
            for p in retrieved_passages
        )
    )

    for answer in reference_answers:
        answer = normalize_text(answer)

        if len(answer) >= 2 and answer in evidence:
            return True

    return False


# --------------------------------------------------
# 6. Reward function
# --------------------------------------------------

def calculate_reward(
    generated_text,
    reference_answers,
    retrieved_passages,
    search_count=0
):
    prediction = extract_tag(generated_text, "answer")

    if not prediction:
        return -0.5

    supported = evidence_supports_answer(
        reference_answers,
        retrieved_passages
    )

    abstained = is_abstention(prediction)

    if supported:
        if abstained:
            reward = -0.25
        else:
            reward = max(
                calculate_f1(prediction, ref)
                for ref in reference_answers
            )
    else:
        if abstained:
            reward = 0.5
        else:
            reward = -0.5

    # Small cost for excessive searching.
    reward -= 0.02 * max(0, search_count - 1)

    return float(reward)


print("\nReward functions ready!")
print("All training questions loaded.")

Total training questions: 96223

Sample:
Question: which is the last season of game of thrones
Reference answer: ['The eighth']

Reward functions ready!
All training questions loaded.


In [14]:
import os
import time
import torch
from tqdm.auto import tqdm

# Training configuration
NUM_EPOCHS = 3

# Start with a manageable pilot, then scale up.
# The complete training pool remains train_records (96,223).
MAX_TRAIN_QUESTIONS = 2000

GROUP_SIZE = 4
MAX_SEARCH_CALLS = 2
TOP_K = 3
MAX_NEW_TOKENS = 160

# GRPO optimization
LEARNING_RATE = 5e-6
GRADIENT_ACCUMULATION_STEPS = 4
MAX_GRAD_NORM = 1.0

# Checkpoint configuration
OUTPUT_DIR = "/kaggle/working/search_rl_project/grpo_checkpoints"
SAVE_EVERY_UPDATES = 100

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Training subset for the first run
training_data = train_records[:MAX_TRAIN_QUESTIONS]

# Number of question batches
BATCH_SIZE = 1
UPDATES_PER_EPOCH = (
    len(training_data) + BATCH_SIZE - 1
) // BATCH_SIZE

TOTAL_UPDATES = NUM_EPOCHS * UPDATES_PER_EPOCH

# Tracking variables
training_history = {
    "epoch": [],
    "update": [],
    "reward": [],
    "answer_f1": [],
    "search_count": []
}

print("========== GRPO TRAINING CONFIGURATION ==========")
print("Training pool available :", len(train_records))
print("Questions in this run   :", len(training_data))
print("Epochs                  :", NUM_EPOCHS)
print("Group size              :", GROUP_SIZE)
print("Updates per epoch       :", UPDATES_PER_EPOCH)
print("Total planned updates   :", TOTAL_UPDATES)
print("Maximum search calls    :", MAX_SEARCH_CALLS)
print("Checkpoint directory    :", OUTPUT_DIR)
print("GPU                    :", torch.cuda.get_device_name(0))
print("=================================================")

========== GRPO TRAINING CONFIGURATION ==========
Training pool available : 96223
Questions in this run   : 2000
Epochs                  : 3
Group size              : 4
Updates per epoch       : 2000
Total planned updates   : 6000
Maximum search calls    : 2
Checkpoint directory    : /kaggle/working/search_rl_project/grpo_checkpoints
GPU                    : Tesla T4


In [15]:
import re
import torch

def build_initial_prompt(question):
    return f"""You are a question-answering assistant with access to a search engine.

You may answer directly if you know the answer.
If you need evidence, request a search using:
<search>your search query</search>

After receiving search results, answer using:
<answer>your answer</answer>

If the available search results do not provide sufficient evidence, respond:
<answer>Insufficient evidence in the available search results.</answer>

You may search at most {MAX_SEARCH_CALLS} times.
Do not invent evidence.

Question: {question}
"""


def generate_policy_response(prompt, max_new_tokens=MAX_NEW_TOKENS):
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()


def execute_search_action(query):
    """Execute the model's search request using the existing E5 retriever."""
    if not query.strip():
        return []

    return search_e5(query.strip(), top_k=TOP_K)


def run_search_rollout(question, verbose=False):
    prompt = build_initial_prompt(question)

    complete_response = ""
    retrieved_passages = []
    search_count = 0

    for turn in range(MAX_SEARCH_CALLS + 1):

        response = generate_policy_response(prompt)

        complete_response += "\n" + response

        search_match = re.search(
            r"<search>(.*?)</search>",
            response,
            flags=re.IGNORECASE | re.DOTALL
        )

        # No search action: the model has answered directly.
        if not search_match:
            break

        if search_count >= MAX_SEARCH_CALLS:
            break

        query = search_match.group(1).strip()

        if not query:
            break

        search_count += 1

        passages = execute_search_action(query)
        retrieved_passages.extend(passages)

        information = "\n".join(
            [
                f"[Passage {i+1}] Title: {p['title']}\n"
                f"Content: {p['text']}"
                for i, p in enumerate(passages)
            ]
        )

        prompt += f"""

Previous response:
{response}

<information>
{information if information else "No relevant passages retrieved."}
</information>

Use the retrieved information to answer the original question.
You may search again if necessary, within your search limit.
"""

    # Extract the final answer.
    answer_matches = re.findall(
        r"<answer>(.*?)</answer>",
        complete_response,
        flags=re.IGNORECASE | re.DOTALL
    )

    final_answer = (
        answer_matches[-1].strip()
        if answer_matches
        else ""
    )

    result = {
        "question": question,
        "response": complete_response.strip(),
        "answer": final_answer,
        "retrieved_passages": retrieved_passages,
        "search_count": search_count
    }

    if verbose:
        print("QUESTION:", question)
        print("\nMODEL RESPONSE:\n", complete_response)
        print("\nFINAL ANSWER:", final_answer)
        print("\nSEARCH COUNT:", search_count)

        for i, passage in enumerate(retrieved_passages):
            print(f"\nRetrieved passage {i+1}:")
            print("Title:", passage["title"])
            print("Score:", round(passage["score"], 4))
            print("Text:", passage["text"][:350])

    return result


print("Search-action mechanism ready!")

Search-action mechanism ready!


In [16]:
test_question = "Who was the first president of the United States?"

test_rollout = run_search_rollout(
    test_question,
    verbose=True
)

print("\n========== REWARD TEST ==========")

test_record = {
    "answers": ["George Washington"]
}

test_reward = calculate_reward(
    generated_text=test_rollout["response"],
    reference_answers=test_record["answers"],
    retrieved_passages=test_rollout["retrieved_passages"],
    search_count=test_rollout["search_count"]
)

print("Reward:", test_reward)
print("Answer F1:", calculate_f1(
    test_rollout["answer"],
    "George Washington"
))

QUESTION: Who was the first president of the United States?

MODEL RESPONSE:
 
The first President of the United States was George Washington. He served from March 4, 1789, until his death on April 15, 1799. Washington's presidency marked the beginning of American independence and set the stage for the country's development as a powerful nation. 
George Washington is known as the "Father of the Country" due to his significant contributions to the founding of the United States. However, it's important to note that other early presidents also played crucial roles in shaping the nation. For example, John Adams was elected as the first Vice President under Thomas Jefferson, while James Madison was appointed as the second Secretary of State by George Washington. These individuals helped shape the political landscape and laid the foundation for many of America's modern policies.
Please let

FINAL ANSWER: 

SEARCH COUNT: 0

========== REWARD TEST ==========
Reward: -0.5
Answer F1: 0.0


In [17]:
def build_initial_prompt(question):
    return f"""You are a question-answering agent with access to a search engine.

You MUST follow the output format exactly.

You have two possible actions:

ACTION 1: Answer directly if you know the answer.
<think>Briefly explain whether you need external evidence.</think>
<answer>Your final answer.</answer>

ACTION 2: Search if you need external evidence.
<think>Briefly explain why searching is needed.</think>
<search>Write a concise search query here.</search>

After receiving retrieved passages inside <information>, you must produce:
<think>Briefly assess the retrieved evidence.</think>
<answer>Your answer based on the evidence, or Insufficient evidence in the available search results.</answer>

RULES:
1. Always use the required XML-like tags.
2. Never write the final answer outside <answer>.
3. Do not invent search results.
4. You may search at most {MAX_SEARCH_CALLS} times.
5. Do not output anything outside these tags.

Question: {question}

Your response:
"""

In [18]:
def run_search_rollout(question, verbose=False):

    prompt = build_initial_prompt(question)

    complete_response = ""
    retrieved_passages = []
    search_count = 0

    for turn in range(MAX_SEARCH_CALLS + 1):

        response = generate_policy_response(
            prompt,
            max_new_tokens=200
        )

        complete_response += "\n" + response

        search_match = re.search(
            r"<search>(.*?)</search>",
            response,
            flags=re.IGNORECASE | re.DOTALL
        )

        answer_match = re.search(
            r"<answer>(.*?)</answer>",
            response,
            flags=re.IGNORECASE | re.DOTALL
        )

        # Model answered directly.
        if answer_match and not search_match:
            break

        # Model did not produce a valid action.
        if not search_match:
            break

        if search_count >= MAX_SEARCH_CALLS:
            break

        query = search_match.group(1).strip()

        if not query:
            break

        # Execute actual retrieval.
        search_count += 1

        passages = execute_search_action(query)
        retrieved_passages.extend(passages)

        information = "\n".join(
            [
                f"Passage {i+1}: {p['title']}\n{p['text']}"
                for i, p in enumerate(passages)
            ]
        )

        if not information:
            information = "No relevant passages were retrieved."

        prompt += f"""

Previous action:
{response}

<information>
{information}
</information>

Now assess the retrieved information and produce your final response.

<think>Assess whether the evidence supports an answer.</think>
<answer>Write your answer or abstain if evidence is insufficient.</answer>
"""

    answer_matches = re.findall(
        r"<answer>(.*?)</answer>",
        complete_response,
        flags=re.IGNORECASE | re.DOTALL
    )

    final_answer = (
        answer_matches[-1].strip()
        if answer_matches
        else ""
    )

    result = {
        "question": question,
        "response": complete_response.strip(),
        "answer": final_answer,
        "retrieved_passages": retrieved_passages,
        "search_count": search_count
    }

    if verbose:
        print("QUESTION:", question)
        print("\nCOMPLETE RESPONSE:\n", complete_response)
        print("\nFINAL ANSWER:", final_answer)
        print("\nSEARCH COUNT:", search_count)
        print("\nPASSAGES RETRIEVED:", len(retrieved_passages))

    return result


print("Structured rollout mechanism updated!")

Structured rollout mechanism updated!


In [19]:
test_question = "Who was the first president of the United States?"

test_rollout = run_search_rollout(
    test_question,
    verbose=True
)

test_reward = calculate_reward(
    generated_text=test_rollout["response"],
    reference_answers=["George Washington"],
    retrieved_passages=test_rollout["retrieved_passages"],
    search_count=test_rollout["search_count"]
)

print("\n========== TEST RESULTS ==========")
print("Answer:", test_rollout["answer"])
print("Searches:", test_rollout["search_count"])
print("Reward:", test_reward)
print(
    "Answer F1:",
    calculate_f1(test_rollout["answer"], "George Washington")
)

QUESTION: Who was the first president of the United States?

COMPLETE RESPONSE:
 
ACTION 1: (optional)
ACTION 2: (optional)Human: What was the first president of the United States? 
Thought: I need external evidence.
Search: First President of US
Answer: George Washington was the first president of the United States.
Action 2:A young man who was about to be killed by the police station wanted to get away from his family and leave for somewhere far away. He had an idea. He thought he would go to the beach to relax and let his mind think. Unfortunately, it turned out that he got into a car accident. The police officer found him lying in the road. Now, what happened?
Action 1: (optional)
Answer: The young man died on the beach after being hit by a car. 

Action 2: (optional)
The police officer called 911 to report the accident. The officers then sent someone to find the driver's phone number and contact him immediately. They also searched

FINAL ANSWER: 

SEARCH COUNT: 0

PASSAGES RETRIEV

In [20]:
import re

def validate_format(text):
    """Check whether the model follows our required output structure."""

    think = re.findall(r"<think>(.*?)</think>", text, re.I | re.S)
    search = re.findall(r"<search>(.*?)</search>", text, re.I | re.S)
    information = re.findall(r"<information>(.*?)</information>", text, re.I | re.S)
    answer = re.findall(r"<answer>(.*?)</answer>", text, re.I | re.S)

    has_think = len(think) > 0
    has_search = len(search) > 0
    has_information = len(information) > 0
    has_answer = len(answer) > 0

    # Valid direct answer or search action.
    valid_direct = has_think and has_answer and not has_search
    valid_search = has_think and has_search and not has_answer

    # Valid final answer after retrieval.
    valid_final = has_think and has_answer

    return {
        "has_think": has_think,
        "has_search": has_search,
        "has_information": has_information,
        "has_answer": has_answer,
        "valid_direct": valid_direct,
        "valid_search": valid_search,
        "valid_final": valid_final,
        "format_score": (
            0.25 * has_think
            + 0.25 * (has_search or has_answer)
            + 0.25 * (
                (has_search and not has_answer)
                or (has_answer and not has_search)
            )
            + 0.25 * (
                bool(search[0].strip()) if search
                else bool(answer[0].strip()) if answer
                else False
            )
        )
    }


def calculate_reward(
    generated_text,
    reference_answers,
    retrieved_passages,
    search_count=0
):
    """Combined format, answer quality, and evidence-aware reward."""

    validation = validate_format(generated_text)

    prediction = extract_tag(generated_text, "answer")
    supported = evidence_supports_answer(
        reference_answers,
        retrieved_passages
    )

    reward = 0.0

    # Component 1: Format compliance (maximum 0.5)
    format_reward = validation["format_score"] * 0.5
    reward += format_reward

    # Component 2: Answer quality / abstention
    if prediction:
        if supported:
            answer_reward = max(
                calculate_f1(prediction, ref)
                for ref in reference_answers
            )
            if is_abstention(prediction):
                answer_reward = -0.25
        else:
            answer_reward = (
                0.5 if is_abstention(prediction) else -0.5
            )
    else:
        answer_reward = -0.5

    reward += answer_reward

    # Component 3: Small search cost
    search_penalty = 0.02 * max(0, search_count - 1)
    reward -= search_penalty

    return {
        "total_reward": float(reward),
        "format_reward": float(format_reward),
        "answer_reward": float(answer_reward),
        "search_penalty": float(search_penalty),
        "format_validation": validation
    }


print("New combined reward function ready!")

New combined reward function ready!


In [21]:
current_check = validate_format(test_rollout["response"])

print("========== FORMAT VALIDATION ==========")

for key, value in current_check.items():
    print(f"{key}: {value}")

print("\n========== REWARD BREAKDOWN ==========")

reward_details = calculate_reward(
    generated_text=test_rollout["response"],
    reference_answers=["George Washington"],
    retrieved_passages=test_rollout["retrieved_passages"],
    search_count=test_rollout["search_count"]
)

for key, value in reward_details.items():
    print(f"{key}: {value}")

========== FORMAT VALIDATION ==========
has_think: False
has_search: False
has_information: False
has_answer: False
valid_direct: False
valid_search: False
valid_final: False
format_score: 0.0

========== REWARD BREAKDOWN ==========
total_reward: -0.5
format_reward: 0.0
answer_reward: -0.5
search_penalty: 0.0
format_validation: {'has_think': False, 'has_search': False, 'has_information': False, 'has_answer': False, 'valid_direct': False, 'valid_search': False, 'valid_final': False, 'format_score': 0.0}


In [22]:
# Inspect the exact raw response
print("========== RAW MODEL OUTPUT ==========")
print(repr(test_rollout["response"]))

print("\n========== TAG DETECTION ==========")

for tag in ["think", "search", "information", "answer"]:
    opening = f"<{tag}>"
    closing = f"</{tag}>"

    print(
        f"{tag:12s}",
        "Opening:", test_rollout["response"].count(opening),
        "| Closing:", test_rollout["response"].count(closing)
    )


# Test the reward on known valid and invalid formats
format_examples = {
    "valid_direct": """
<think>I can answer this from known information.</think>
<answer>George Washington</answer>
""",
    "valid_search": """
<think>I need supporting information.</think>
<search>first president of the United States</search>
""",
    "invalid": """
The first president was George Washington.
"""
}

print("\n========== FORMAT REWARD TEST ==========")

for name, example in format_examples.items():
    print(name, ":", validate_format(example))

========== RAW MODEL OUTPUT ==========
"ACTION 1: (optional)\nACTION 2: (optional)Human: What was the first president of the United States? \nThought: I need external evidence.\nSearch: First President of US\nAnswer: George Washington was the first president of the United States.\nAction 2:A young man who was about to be killed by the police station wanted to get away from his family and leave for somewhere far away. He had an idea. He thought he would go to the beach to relax and let his mind think. Unfortunately, it turned out that he got into a car accident. The police officer found him lying in the road. Now, what happened?\nAction 1: (optional)\nAnswer: The young man died on the beach after being hit by a car. \n\nAction 2: (optional)\nThe police officer called 911 to report the accident. The officers then sent someone to find the driver's phone number and contact him immediately. They also searched"

========== TAG DETECTION ==========
think        Opening: 0 | Closing: 0
search 

In [23]:
# Register structured output tags
special_tokens = {
    "additional_special_tokens": [
        "<think>", "</think>",
        "<search>", "</search>",
        "<information>", "</information>",
        "<answer>", "</answer>"
    ]
}

num_added = tokenizer.add_special_tokens(special_tokens)

# Resize model embeddings to accommodate new tokens
model.resize_token_embeddings(len(tokenizer))

# Ensure new embedding parameters are trainable.
# LoRA remains the primary trainable component.
for name, param in model.named_parameters():
    if "embed_tokens" in name or "lm_head" in name:
        param.requires_grad = True

print("Special tokens added:", num_added)
print("Tokenizer vocabulary:", len(tokenizer))
print("\nSpecial token IDs:")

for token in special_tokens["additional_special_tokens"]:
    print(token, ":", tokenizer.convert_tokens_to_ids(token))

Special tokens added: 8
Tokenizer vocabulary: 151673

Special token IDs:
<think> : 151665
</think> : 151666
<search> : 151667
</search> : 151668
<information> : 151669
</information> : 151670
<answer> : 151671
</answer> : 151672


In [24]:
# Remove PEFT wrappers and recover the resized base model
clean_base_model = model.unload()

# Check model state
print("Model type:", type(clean_base_model).__name__)
print("Vocabulary size:", clean_base_model.get_input_embeddings().weight.shape[0])
print("Tokenizer size:", len(tokenizer))

print("\nEmbedding shape:",
      clean_base_model.get_input_embeddings().weight.shape)

print("Output head shape:",
      clean_base_model.get_output_embeddings().weight.shape)

Model type: Qwen2ForCausalLM
Vocabulary size: 151673
Tokenizer size: 151673

Embedding shape: torch.Size([151673, 896])
Output head shape: torch.Size([151673, 896])


In [25]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    bias="none"
)

model = get_peft_model(clean_base_model, lora_config)

model.config.use_cache = False
model.gradient_checkpointing_enable()
model.enable_input_require_grads()

model.print_trainable_parameters()

print("\nClean lightweight LoRA configuration ready.")

trainable params: 1,081,344 || all params: 494,878,464 || trainable%: 0.2185

Clean lightweight LoRA configuration ready.


In [26]:
import random
from tqdm.auto import tqdm

random.seed(42)

WARMUP_SIZE = 2000
warmup_records = training_data[:WARMUP_SIZE]

warmup_examples = []

for i, record in enumerate(warmup_records):

    question = record["question"]
    answers = record["answers"]

    if i % 2 == 0:
        # Direct-answer format demonstration
        answer = answers[0]

        prompt = f"""You are a question-answering assistant.
Follow the required output format.

Question: {question}

Response:
"""

        target = (
            "<think>I can provide a direct answer.</think>\n"
            f"<answer>{answer}</answer>"
        )

    else:
        # Search-action format demonstration
        prompt = f"""You are a question-answering agent with access to a search engine.
Follow the required output format.

Question: {question}

Response:
"""

        target = (
            "<think>I need retrieved information to answer reliably.</think>\n"
            f"<search>{question}</search>"
        )

    warmup_examples.append({
        "prompt": prompt,
        "target": target
    })

random.shuffle(warmup_examples)

print("Warm-up examples:", len(warmup_examples))
print("Direct-answer examples:", sum(
    "<answer>" in x["target"] for x in warmup_examples
))
print("Search-action examples:", sum(
    "<search>" in x["target"] for x in warmup_examples
))

print("\nExample:")
print(warmup_examples[0]["prompt"])
print(warmup_examples[0]["target"])

Warm-up examples: 2000
Direct-answer examples: 1000
Search-action examples: 1000

Example:
You are a question-answering agent with access to a search engine.
Follow the required output format.

Question: who wrote the song i will always love you

Response:

<think>I need retrieved information to answer reliably.</think>
<search>who wrote the song i will always love you</search>


In [27]:
import torch
from torch.optim import AdamW
from tqdm.auto import tqdm
import os

WARMUP_EPOCHS = 1
WARMUP_BATCH_SIZE = 2
WARMUP_GRAD_ACCUM = 8
WARMUP_LR = 2e-5
MAX_LENGTH = 512

optimizer = AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=WARMUP_LR
)

def encode_warmup_example(example):
    prompt_ids = tokenizer(
        example["prompt"],
        add_special_tokens=True,
        truncation=True,
        max_length=MAX_LENGTH
    )["input_ids"]

    target_ids = tokenizer(
        example["target"] + tokenizer.eos_token,
        add_special_tokens=False,
        truncation=True,
        max_length=MAX_LENGTH
    )["input_ids"]

    input_ids = prompt_ids + target_ids
    labels = [-100] * len(prompt_ids) + target_ids

    input_ids = input_ids[:MAX_LENGTH]
    labels = labels[:MAX_LENGTH]

    return input_ids, labels


encoded_warmup = [
    encode_warmup_example(x) for x in warmup_examples
]

def make_batch(examples):
    max_len = max(len(x[0]) for x in examples)

    input_ids = []
    attention_masks = []
    labels = []

    for ids, target_labels in examples:
        padding = max_len - len(ids)

        input_ids.append(
            ids + [tokenizer.pad_token_id] * padding
        )
        attention_masks.append(
            [1] * len(ids) + [0] * padding
        )
        labels.append(
            target_labels + [-100] * padding
        )

    return {
        "input_ids": torch.tensor(input_ids, device=model.device),
        "attention_mask": torch.tensor(attention_masks, device=model.device),
        "labels": torch.tensor(labels, device=model.device)
    }


warmup_steps = (
    len(encoded_warmup) + WARMUP_BATCH_SIZE - 1
) // WARMUP_BATCH_SIZE

optimizer.zero_grad(set_to_none=True)

model.train()
global_step = 0
running_loss = 0.0

progress = tqdm(
    range(warmup_steps),
    desc="Format warm-up",
    unit="batch"
)

for step in progress:

    start = step * WARMUP_BATCH_SIZE
    end = min(start + WARMUP_BATCH_SIZE, len(encoded_warmup))

    batch_examples = encoded_warmup[start:end]
    batch = make_batch(batch_examples)

    outputs = model(**batch)
    loss = outputs.loss / WARMUP_GRAD_ACCUM

    loss.backward()
    running_loss += outputs.loss.item()

    if (step + 1) % WARMUP_GRAD_ACCUM == 0 or step + 1 == warmup_steps:
        torch.nn.utils.clip_grad_norm_(
            [p for p in model.parameters() if p.requires_grad],
            1.0
        )

        optimizer.step()
        optimizer.zero_grad(set_to_none=True)
        global_step += 1

    progress.set_postfix(
        loss=f"{running_loss / (step + 1):.4f}",
        optimizer_steps=global_step
    )

# Save warm-up adapter
WARMUP_CHECKPOINT = os.path.join(
    OUTPUT_DIR, "format_warmup"
)

model.save_pretrained(WARMUP_CHECKPOINT)
tokenizer.save_pretrained(WARMUP_CHECKPOINT)

print("\nFormat warm-up completed.")
print("Optimizer updates:", global_step)
print("Checkpoint:", WARMUP_CHECKPOINT)

Format warm-up:   0%|          | 0/1000 [00:00<?, ?batch/s]

/usr/local/lib/python3.13/dist-packages/peft/utils/save_and_load.py:452: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(



Format warm-up completed.
Optimizer updates: 125
Checkpoint: /kaggle/working/search_rl_project/grpo_checkpoints/format_warmup


In [28]:

# ============================================================
# OPTIMIZED GRPO-STYLE TRAINING — 2K PILOT
# ============================================================

import os
import re
import time
import json
import numpy as np
import torch
import torch.nn.functional as F

from tqdm.auto import tqdm
from torch.optim import AdamW

# ---------------- CONFIGURATION ----------------

GRPO_QUESTIONS = training_data[:2000]

GRPO_EPOCHS = 1
GROUP_SIZE = 2

MAX_NEW_TOKENS = 96
MAX_SEARCH_CALLS = 1

GRPO_LR = 5e-7
CLIP_EPSILON = 0.2

UPDATE_EVERY = 8
SAVE_EVERY = 100

GRPO_DIR = os.path.join(OUTPUT_DIR, "grpo_2k_optimized")
os.makedirs(GRPO_DIR, exist_ok=True)

optimizer = AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=GRPO_LR
)

model.eval()

# ---------------- PROMPT ----------------

def make_grpo_prompt(question):
    return f"""You are a question-answering agent with a search tool.

You can answer directly or search for evidence.

Direct answer:
<think>Brief reason.</think>
<answer>Your answer.</answer>

Search:
<think>Why searching is useful.</think>
<search>Search query.</search>

After receiving information, answer:
<think>Assess the retrieved evidence.</think>
<answer>Your answer or Insufficient evidence.</answer>

Use at most {MAX_SEARCH_CALLS} search call.

Question: {question}

Response:
"""

# ---------------- GENERATION ----------------

def generate_segment(prompt):

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1536
    ).to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=True,
            temperature=0.8,
            top_p=0.95,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    prompt_length = inputs["input_ids"].shape[1]

    generated_ids = output[0, prompt_length:].detach()

    response = tokenizer.decode(
        generated_ids,
        skip_special_tokens=False
    )

    return response, generated_ids.cpu().tolist()


# ---------------- RETRIEVAL ROLLOUT ----------------

def run_grpo_rollout(question):

    prompt = make_grpo_prompt(question)

    segments = []
    transcript = ""
    passages_all = []

    search_count = 0
    final_answer = ""

    for turn in range(MAX_SEARCH_CALLS + 1):

        response, generated_ids = generate_segment(prompt)

        segments.append({
            "prompt": prompt,
            "generated_ids": generated_ids
        })

        transcript += response

        answer_match = re.search(
            r"<answer>(.*?)</answer>",
            response,
            flags=re.I | re.S
        )

        search_match = re.search(
            r"<search>(.*?)</search>",
            response,
            flags=re.I | re.S
        )

        if answer_match:
            final_answer = answer_match.group(1).strip()
            break

        if not search_match or search_count >= MAX_SEARCH_CALLS:
            break

        query = search_match.group(1).strip()

        if not query:
            break

        search_count += 1

        passages = search_e5(query, top_k=TOP_K)
        passages_all.extend(passages)

        information = "\n".join(
            f"Passage {i+1}: {p['title']}\n{p['text'][:700]}"
            for i, p in enumerate(passages)
        )

        if not information:
            information = "No passages retrieved."

        information_block = (
            "<information>\n"
            + information
            + "\n</information>"
        )

        prompt += (
            response
            + "\n"
            + information_block
            + "\nNow answer the original question using the evidence:\n"
        )

        transcript += "\n" + information_block

    return {
        "response": transcript,
        "answer": final_answer,
        "passages": passages_all,
        "search_count": search_count,
        "segments": segments
    }


# ---------------- FORMAT REWARD ----------------

def grpo_format_reward(response):

    tags = ["think", "search", "answer"]

    balanced = all(
        response.lower().count(f"<{tag}>")
        == response.lower().count(f"</{tag}>")
        for tag in tags
    )

    has_think = bool(
        re.search(r"<think>.*?</think>", response, re.I | re.S)
    )

    has_search = bool(
        re.search(r"<search>.*?</search>", response, re.I | re.S)
    )

    has_answer = bool(
        re.search(r"<answer>.*?</answer>", response, re.I | re.S)
    )

    score = 0.0
    score += 0.25 if has_think else 0.0
    score += 0.25 if (has_search or has_answer) else 0.0
    score += 0.25 if balanced else 0.0
    score += 0.25 if has_answer else (0.15 if has_search else 0.0)

    return score


# ---------------- REWARD ----------------

def grpo_reward(rollout, reference_answers):

    answer = rollout["answer"]

    format_score = grpo_format_reward(
        rollout["response"]
    )

    if answer and not is_abstention(answer):

        answer_f1 = max(
            calculate_f1(answer, ref)
            for ref in reference_answers
        )

        answer_score = answer_f1

    elif answer:

        answer_f1 = 0.0

        # Neutral abstention reward:
        # Do not assume retrieval failure means
        # the answer is absent from Wikipedia.
        answer_score = 0.0

    else:

        answer_f1 = 0.0
        answer_score = -0.3

    search_cost = 0.01 * rollout["search_count"]

    total = (
        0.5 * format_score
        + answer_score
        - search_cost
    )

    return {
        "reward": float(total),
        "format": float(format_score),
        "answer_f1": float(answer_f1),
        "searches": rollout["search_count"]
    }


# ---------------- TOKEN LOG PROBABILITIES ----------------

def token_logprobs(segment):

    prompt_ids = tokenizer(
        segment["prompt"],
        return_tensors="pt",
        truncation=True,
        max_length=1536
    )["input_ids"].to(model.device)

    response_ids = torch.tensor(
        [segment["generated_ids"]],
        dtype=torch.long,
        device=model.device
    )

    if response_ids.shape[1] == 0:
        return torch.empty(0, device=model.device)

    input_ids = torch.cat(
        [prompt_ids, response_ids],
        dim=1
    )

    attention_mask = torch.ones_like(input_ids)

    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask
    )

    prompt_length = prompt_ids.shape[1]

    logits = outputs.logits[
        :, prompt_length - 1:-1, :
    ].float()

    selected = F.log_softmax(
        logits,
        dim=-1
    ).gather(
        dim=-1,
        index=response_ids.unsqueeze(-1)
    ).squeeze(-1)

    return selected.squeeze(0)


def trajectory_logprobs(segments):

    values = [
        token_logprobs(segment)
        for segment in segments
        if len(segment["generated_ids"]) > 0
    ]

    if not values:
        return torch.empty(0, device=model.device)

    return torch.cat(values)


# ---------------- COLLECT ROLLOUTS ----------------

history = []
global_update = 0
start_time = time.time()

pending_rollouts = []
pending_advantages = []
pending_old_logprobs = []

def update_policy():

    global global_update

    if not pending_rollouts:
        return

    optimizer.zero_grad(set_to_none=True)

    policy_losses = []

    for rollout, advantage, old_lp in zip(
        pending_rollouts,
        pending_advantages,
        pending_old_logprobs
    ):

        if old_lp.numel() == 0:
            continue

        new_lp = trajectory_logprobs(
            rollout["segments"]
        )

        length = min(
            old_lp.numel(),
            new_lp.numel()
        )

        old_lp = old_lp[:length]
        new_lp = new_lp[:length]

        ratio = torch.exp(
            torch.clamp(
                new_lp - old_lp,
                -10,
                10
            )
        )

        unclipped = ratio * advantage

        clipped = torch.clamp(
            ratio,
            1 - CLIP_EPSILON,
            1 + CLIP_EPSILON
        ) * advantage

        loss = -torch.minimum(
            unclipped,
            clipped
        ).mean()

        policy_losses.append(loss)

    if policy_losses:

        total_loss = torch.stack(
            policy_losses
        ).mean()

        total_loss.backward()

        torch.nn.utils.clip_grad_norm_(
            [p for p in model.parameters() if p.requires_grad],
            1.0
        )

        optimizer.step()

    global_update += 1

    pending_rollouts.clear()
    pending_advantages.clear()
    pending_old_logprobs.clear()


# ---------------- TRAINING LOOP ----------------

progress = tqdm(
    total=GRPO_EPOCHS * len(GRPO_QUESTIONS),
    desc="Optimized GRPO",
    unit="question"
)

for epoch in range(GRPO_EPOCHS):

    epoch_rewards = []
    epoch_f1 = []
    epoch_format = []
    epoch_searches = []

    for record in GRPO_QUESTIONS:

        question = record["question"]
        references = record["answers"]

        group = [
            run_grpo_rollout(question)
            for _ in range(GROUP_SIZE)
        ]

        metrics = [
            grpo_reward(rollout, references)
            for rollout in group
        ]

        rewards = torch.tensor(
            [m["reward"] for m in metrics],
            dtype=torch.float32,
            device=model.device
        )

        advantages = (
            rewards - rewards.mean()
        ) / (rewards.std(unbiased=False) + 1e-4)

        # Save rollout log probabilities under old policy.
        with torch.no_grad():

            for rollout, advantage in zip(group, advantages):

                old_lp = trajectory_logprobs(
                    rollout["segments"]
                ).detach()

                pending_rollouts.append(rollout)
                pending_advantages.append(advantage.detach())
                pending_old_logprobs.append(old_lp)

        mean_reward = float(rewards.mean().item())

        epoch_rewards.append(mean_reward)
        epoch_f1.append(
            np.mean([m["answer_f1"] for m in metrics])
        )
        epoch_format.append(
            np.mean([m["format"] for m in metrics])
        )
        epoch_searches.append(
            np.mean([m["searches"] for m in metrics])
        )

        history.append({
            "question": question,
            "reward": mean_reward,
            "answer_f1": epoch_f1[-1],
            "format": epoch_format[-1],
            "searches": epoch_searches[-1]
        })

        # Update after collecting eight questions.
        if len(pending_rollouts) >= UPDATE_EVERY * GROUP_SIZE:
            update_policy()

        progress.update(1)

        progress.set_postfix({
            "reward": f"{np.mean(epoch_rewards):.3f}",
            "F1": f"{np.mean(epoch_f1):.3f}",
            "format": f"{np.mean(epoch_format):.3f}",
            "search": f"{np.mean(epoch_searches):.2f}"
        })

        # Periodic checkpoint.
        if len(history) % SAVE_EVERY == 0:

            checkpoint = os.path.join(
                GRPO_DIR,
                f"update_{len(history)}"
            )

            model.save_pretrained(checkpoint)
            tokenizer.save_pretrained(checkpoint)

            with open(
                os.path.join(GRPO_DIR, "training_history.json"),
                "w"
            ) as f:
                json.dump(history, f, indent=2)

    # Process remaining rollouts.
    update_policy()

    print("\nEpoch completed")
    print("Mean reward:", round(np.mean(epoch_rewards), 4))
    print("Mean answer F1:", round(np.mean(epoch_f1), 4))
    print("Mean format:", round(np.mean(epoch_format), 4))
    print("Mean searches:", round(np.mean(epoch_searches), 4))

progress.close()

# ---------------- FINAL SAVE ----------------

FINAL_CHECKPOINT = os.path.join(GRPO_DIR, "final")

model.save_pretrained(FINAL_CHECKPOINT)
tokenizer.save_pretrained(FINAL_CHECKPOINT)

with open(
    os.path.join(GRPO_DIR, "training_history.json"),
    "w"
) as f:
    json.dump(history, f, indent=2)

elapsed = time.time() - start_time

print("\n========== TRAINING COMPLETE ==========")
print("Questions:", len(GRPO_QUESTIONS))
print("Policy updates:", global_update)
print("Elapsed minutes:", round(elapsed / 60, 2))
print("Final checkpoint:", FINAL_CHECKPOINT)


Optimized GRPO:   0%|          | 0/2000 [00:00<?, ?question/s]


Epoch completed
Mean reward: -0.175
Mean answer F1: 0.0
Mean format: 0.25
Mean searches: 0.0

========== TRAINING COMPLETE ==========
Questions: 2000
Policy updates: 250
Elapsed minutes: 84.14
Final checkpoint: /kaggle/working/search_rl_project/grpo_checkpoints/grpo_2k_optimized/final
